# Step 1: Foundations, setup and baseline

**Goal for this notebook (Nov 9 to 13):** understand *why* LLM inference is fast or slow, then measure a reference model on your GPU and check that the numbers make sense.

By the end you will have:
1. A working PyTorch + CUDA setup on your RTX 5070 (Blackwell, `sm_120`).
2. Your GPU's two key numbers (memory bandwidth, peak FLOPs) and its *ridge point*.
3. Formulas you derived yourself for parameter count, FLOPs per token, and KV-cache size, each checked against the real model.
4. A written **prediction** of decode speed, then a **measured** baseline (time to first token, decode tokens/sec, peak memory) with mean and spread.
5. The same benchmark on Modal, with cold start and cost per million tokens.

### How this notebook works
- 📖 **Explanation cells** teach a concept right before you need it.
- ✏️ **TODO cells** are yours. They contain a signature and `raise NotImplementedError`. Replace that with your code.
- ✅ **Check cells** (marked `# CHECK`) test your work. Don't edit them; if one fails, read the message, fix your TODO, re-run.
- 💡 **Hints** are hidden in collapsible blocks, in escalating order. Try without them first. Open hint 1, try again, then hint 2.
- ❓ **Quiz** at the end. Answer in your own words before moving on to Step 2.

If you're stuck after the hints, ask in the project thread with what you tried and what you saw.

## 0. Setup (do this once, outside the notebook)

1. **Driver:** you need a recent NVIDIA driver that supports CUDA 12.8+. Run `nvidia-smi` in a terminal; the top right shows the max CUDA version the driver supports.
2. **Environment:** make a fresh virtual environment (venv or conda) for this project.
3. **PyTorch built for CUDA 12.8** (older wheels don't know about Blackwell):
   ```
   pip install torch --index-url https://download.pytorch.org/whl/cu128
   ```
4. **Other packages:**
   ```
   pip install "transformers>=4.51" accelerate pandas jupyter modal
   ```
   (Qwen3 support landed in transformers 4.51.)
5. **Repo:** create a new GitHub repo (e.g. `mini-inference-engine`), clone it, and put this notebook inside it. Suggested layout:
   ```
   engine/      # your engine code (Step 2 onward)
   bench/       # benchmark scripts, including the Modal one
   tests/       # correctness tests vs the reference
   results/     # CSVs of every benchmark run
   notebooks/   # these notebooks
   README.md    # goal + the running benchmark table
   ```
6. **Benchmark hygiene, from day one:** laptop plugged in, same power mode every time, close other GPU apps (browsers use the GPU!).

## 1. Environment check

In [ ]:
# CHECK: environment
import sys, platform, torch
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("CUDA (torch):", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA not available: check the driver and that you installed the cu128 wheel."
print("GPU         :", torch.cuda.get_device_name(0))
print("capability  :", torch.cuda.get_device_capability(0))
archs = torch.cuda.get_arch_list()
print("arch list   :", archs)
assert any(a in archs for a in ("sm_120", "compute_120")), (
    "This PyTorch build has no sm_120 kernels. Reinstall from the cu128 index URL above.")
x = torch.randn(1024, 1024, device="cuda", dtype=torch.bfloat16)
assert torch.isfinite(x @ x).all()
print("✅ environment OK")

## 2. Your GPU's two numbers

📖 Almost every performance question in this project comes down to two hardware numbers:

- **Peak compute** (FLOP/s): how much arithmetic the GPU can do per second. For us the relevant number is dense **BF16/FP16 tensor core** throughput (matmuls run on tensor cores).
- **Memory bandwidth** (bytes/s): how fast the GPU can move data between its DRAM (the 8 GB of GDDR7) and the compute units.

Their ratio is the **ridge point** (also called the critical arithmetic intensity):

$$\text{ridge} = \frac{\text{peak FLOP/s}}{\text{bandwidth (bytes/s)}} \quad [\text{FLOPs per byte}]$$

If a piece of work does fewer FLOPs per byte than this, it can't keep the compute units busy: it's **memory-bound**. If it does more, it's **compute-bound**.

✏️ **Your job:** look these up for *your* GPU (RTX 5070 **Laptop**, not the desktop card; laptop specs vary with power limit, so note the source). Good places: NVIDIA's spec pages, the Blackwell architecture whitepaper (tensor core throughput tables), TechPowerUp's GPU database. For bandwidth: bandwidth = (memory data rate per pin) × (bus width in bits) / 8.

Be careful with the FLOPs number: spec sheets often quote **sparse** or **FP4/FP8** numbers. You want **dense BF16/FP16** with FP32 accumulate.

In [ ]:
# TODO: fill these in from your research. Write the source next to each number.
BANDWIDTH_GBPS = None        # GB/s (1 GB = 1e9 bytes)        source:
PEAK_BF16_TFLOPS = None      # dense BF16 tensor TFLOP/s       source:

def ridge_point(peak_tflops: float, bandwidth_gbps: float) -> float:
    # Return FLOPs per byte.
    raise NotImplementedError

<details><summary>💡 Hint 1</summary>
Watch your units. TFLOP/s is 1e12 FLOP/s; GB/s is 1e9 bytes/s.
</details>

In [ ]:
# CHECK: GPU numbers are plausible
assert BANDWIDTH_GBPS is not None and PEAK_BF16_TFLOPS is not None, "Fill in both numbers above."
assert 200 <= BANDWIDTH_GBPS <= 1000, f"{BANDWIDTH_GBPS} GB/s looks off for a laptop GPU. Recheck bus width and data rate."
assert 10 <= PEAK_BF16_TFLOPS <= 300, f"{PEAK_BF16_TFLOPS} TFLOP/s looks off. Did you grab a sparse or FP4 number?"
r = ridge_point(PEAK_BF16_TFLOPS, BANDWIDTH_GBPS)
assert abs(r - PEAK_BF16_TFLOPS * 1e12 / (BANDWIDTH_GBPS * 1e9)) < 1e-6 * r, "Check your unit conversion."
print(f"✅ ridge point ≈ {r:.0f} FLOPs/byte")

## 3. Arithmetic intensity and the roofline

📖 Read first: *How to Scale Your Model*, chapter **"All About Rooflines"**. Skip TPU-specific details.

**Arithmetic intensity** of an operation = FLOPs it performs ÷ bytes it must move to/from memory.

**Roofline model:** time for an operation is roughly

$$t \approx \max\left(\frac{\text{FLOPs}}{\text{peak FLOP/s}},\ \frac{\text{bytes}}{\text{bandwidth}}\right)$$

whichever limit you hit first. (Real kernels don't overlap perfectly, so this is a *lower bound* on time.)

The most important operation in a transformer is a matrix multiply: an input of shape `(M, K)` times a weight of shape `(K, N)` gives `(M, N)`. In inference, **M is the number of tokens being processed at once** (prompt length during prefill, batch size during decode).

✏️ Write the FLOP count and byte count for this matmul, assuming every element is 2 bytes (bf16), each input is read once and the output is written once.

In [ ]:
def matmul_flops(M: int, K: int, N: int) -> int:
    raise NotImplementedError

def matmul_bytes(M: int, K: int, N: int, bytes_per_elem: int = 2) -> int:
    raise NotImplementedError

def matmul_intensity(M: int, K: int, N: int, bytes_per_elem: int = 2) -> float:
    raise NotImplementedError

<details><summary>💡 Hint 1</summary>
Each output element is a dot product of length K. How many multiplies and adds is that?
</details>
<details><summary>💡 Hint 2</summary>
A multiply-add counts as 2 FLOPs. Bytes: three matrices touch memory (two read, one written).
</details>

In [ ]:
# CHECK: matmul intensity
assert matmul_flops(2, 3, 4) == 48, "FLOPs for (2,3)@(3,4) should be 48."
assert matmul_bytes(2, 3, 4) == 2 * (6 + 12 + 8)
assert abs(matmul_intensity(4096, 4096, 4096) - 1365.33) < 0.1
assert abs(matmul_intensity(1, 4096, 4096) - 0.9995) < 1e-3
print("✅ matmul math correct\n")

r = ridge_point(PEAK_BF16_TFLOPS, BANDWIDTH_GBPS)
for M in [1, 8, 64, 256, 1024, 4096]:
    ai = matmul_intensity(M, 1024, 3072)
    print(f"M={M:5d} tokens  intensity={ai:8.1f} FLOPs/byte  ->  {'COMPUTE' if ai > r else 'MEMORY'}-bound")

✍️ **Stop and write (in this cell, replace the text):** looking at the table above, at roughly how many tokens processed at once does a matmul in this model switch from memory-bound to compute-bound on your GPU? What does that tell you about decode at batch size 1 vs prefill of a 512-token prompt?

*Your answer:*

## 4. Transformer math for your model

📖 Read first: *How to Scale Your Model*, chapter **"All the Transformer Math You Need"** (the inference-relevant parts: parameter counting, FLOPs per token, KV cache). Skip training/backward-pass and sharding sections.

We'll use **Qwen3-0.6B** (if a newer small Qwen release with the same architecture exists, it's fine to use it, but stick with one for the whole project). Later, Qwen3-1.7B can be the target and 0.6B the draft for speculative decoding.

First, load the model and look at its structure. Read `print(model)` carefully: every module listed there is something you'll implement yourself in Step 2.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_ID = "Qwen/Qwen3-0.6B"
DTYPE = torch.bfloat16

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to("cuda").eval()
cfg = model.config
print(model)
print({k: getattr(cfg, k) for k in ["vocab_size", "hidden_size", "intermediate_size", "num_hidden_layers",
                                     "num_attention_heads", "num_key_value_heads", "head_dim", "tie_word_embeddings"]})

📖 Things to notice in the printout:
- **Grouped-query attention (GQA):** `num_key_value_heads` < `num_attention_heads`. Several query heads share one K/V head. This shrinks the KV cache. Why does that matter for memory?
- **`q_norm` / `k_norm`:** Qwen3 applies RMSNorm to queries and keys per head. Small, but they're parameters too.
- **Gated MLP:** three projections (`gate_proj`, `up_proj`, `down_proj`), not two.
- **`tie_word_embeddings`:** if true, the LM head reuses the embedding matrix, so it's stored once.

### 4a. Parameter count
✏️ Write a function that computes the total number of parameters **from the config alone**. Go module by module through `print(model)`.

In [ ]:
def count_params(cfg) -> int:
    raise NotImplementedError

<details><summary>💡 Hint 1</summary>
Split it into: embeddings, per-layer (attention projections, attention norms, MLP, two layer norms) × num layers, final norm, LM head (only if not tied). Does this model have biases on its linear layers? Check the printout.
</details>
<details><summary>💡 Hint 2</summary>
q_proj maps hidden_size → num_attention_heads × head_dim. k_proj and v_proj map hidden_size → num_key_value_heads × head_dim. o_proj maps back. An RMSNorm over d features has d parameters.
</details>

In [ ]:
# CHECK: parameter count matches the real model
actual = sum(p.numel() for p in model.parameters())   # shared (tied) tensors are counted once
mine = count_params(cfg)
print(f"yours: {mine:,}   actual: {actual:,}")
assert mine == actual, f"Off by {mine - actual:,}. Compare against: {[(n, p.numel()) for n, p in list(model.named_parameters())[:12]]}"
print("✅ parameter count exact")

### 4b. FLOPs and bytes per token

📖 Rule of thumb from the reading: a forward pass costs about **2 × (parameters) FLOPs per token** (every weight is used in one multiply-add per token), plus attention's extra work that grows with context length. For short contexts and a small model, the 2N term dominates.

During **decode at batch size 1**, every step must read **every weight from memory once** to produce **one** token.

✏️ Fill in the approximate numbers.

In [ ]:
def weight_bytes(cfg, bytes_per_param: int = 2) -> int:
    raise NotImplementedError

def flops_per_token(cfg) -> float:
    # Use the 2 × params approximation (ignore attention-score FLOPs for now).
    raise NotImplementedError

<details><summary>💡 Hint 1</summary>
Should the embedding lookup count toward FLOPs? An embedding is a table lookup, not a matmul. The LM head (even if tied) IS a matmul. Think about which weights actually do multiply-adds. Either answer is OK for a rough estimate as long as you can say why.
</details>

In [ ]:
# CHECK: weights and FLOPs
wb = weight_bytes(cfg)
torch_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
assert abs(wb - torch_bytes) / torch_bytes < 0.01, f"weight_bytes={wb:,} but model holds {torch_bytes:,} bytes."
f = flops_per_token(cfg)
assert 0.6e9 <= f <= 1.6e9, f"{f:.3g} FLOPs/token is outside the plausible range for this model."
print(f"✅ weights: {wb/1e9:.2f} GB   FLOPs/token ≈ {f/1e9:.2f} GFLOP   intensity at batch 1 ≈ {f/wb:.2f} FLOPs/byte")

### 4c. KV cache size

📖 During generation, each layer stores the **keys and values** of every previous token so it doesn't recompute them. That's the KV cache. Its size grows with sequence length and batch size, and it competes with the weights for your 8 GB.

✏️ Write the number of bytes the KV cache needs **per token** (batch size 1, all layers).

In [ ]:
def kv_bytes_per_token(cfg, bytes_per_elem: int = 2) -> int:
    raise NotImplementedError

<details><summary>💡 Hint 1</summary>
Per layer, per token: one key vector and one value vector for each <b>KV</b> head (not query head). How long is each vector?
</details>

In [ ]:
# CHECK: KV cache size vs what Hugging Face actually allocates
def _cache_bytes(cache):
    if hasattr(cache, "layers"):                      # transformers >= 4.56
        kv = [(l.keys, l.values) for l in cache.layers]
    elif hasattr(cache, "key_cache"):                 # older DynamicCache
        kv = list(zip(cache.key_cache, cache.value_cache))
    else:                                             # legacy tuple format
        kv = list(cache)
    return sum(t.numel() * t.element_size() for pair in kv for t in pair)

L = 100
ids = torch.randint(0, cfg.vocab_size, (1, L), device="cuda")
with torch.no_grad():
    out = model(ids, use_cache=True)
measured = _cache_bytes(out.past_key_values) / L
mine = kv_bytes_per_token(cfg)
print(f"yours: {mine:,} B/token   HF: {measured:,.0f} B/token")
assert mine == measured, "Mismatch. Did you use num_key_value_heads, and count both K and V?"
print(f"✅ KV cache per token = {mine/1024:.0f} KiB")
free_gb = 8 - wb / 1e9 - 1.0   # leave ~1 GB for activations, CUDA context, etc.
print(f"   with ~{free_gb:.1f} GB free, you could hold ≈ {free_gb*1e9/mine:,.0f} tokens of KV cache total (across the whole batch)")
del out

## 5. Predict before you measure

📖 This is the habit interviewers care most about: **write the prediction first**, then measure, then explain the gap.

Using your numbers above:
- **Decode at batch 1:** each token requires reading all weights once. The roofline lower bound on time per token is `max(flops/peak, bytes/bandwidth)`. Which term wins?
- **Prefill of a P-token prompt:** the weights are read once but used for P tokens. For P = 512, which term wins?

✏️ Fill in your predictions. Don't run the benchmark before you do.

In [ ]:
PROMPT_LEN = 512
NEW_TOKENS = 128

def predict_decode_ms_per_token(cfg) -> float:
    raise NotImplementedError

def predict_prefill_ms(cfg, prompt_len: int) -> float:
    raise NotImplementedError

pred_decode = predict_decode_ms_per_token(cfg)
pred_prefill = predict_prefill_ms(cfg, PROMPT_LEN)
print(f"PREDICTION  decode: {pred_decode:.2f} ms/token  ({1000/pred_decode:.0f} tok/s)   prefill({PROMPT_LEN}): {pred_prefill:.2f} ms")

✍️ **Write down now (before measuring):** do you expect the real Hugging Face numbers to be close to these? Faster or slower? Why?

*Your answer:*

## 6. Measure the baseline

📖 **How to time GPU code correctly.** CUDA launches are *asynchronous*: Python returns immediately while the GPU keeps working. If you use `time.time()` without synchronizing, you measure how long it took to *queue* the work, not to do it. Two correct options:
- `torch.cuda.synchronize()` before starting and before stopping a `time.perf_counter()` timer, or
- `torch.cuda.Event(enable_timing=True)` pairs: `start.record()`, run, `end.record()`, `torch.cuda.synchronize()`, `start.elapsed_time(end)` (in ms).

📖 **Warmup.** The first runs are slow (CUDA context init, kernel selection, caches, clocks ramping up). Always run a few warmup iterations and throw them away.

📖 **Spread.** One number means nothing. Run several times; report mean and standard deviation.

📖 **What to measure:**
- **TTFT (time to first token):** prompt in → first new token out. Mostly prefill.
- **Decode time per token** and **decode tokens/sec** (excluding the first token).
- **Peak GPU memory:** `torch.cuda.reset_peak_memory_stats()` before, `torch.cuda.max_memory_allocated()` after.

Use greedy decoding (`do_sample=False`) and force exactly `NEW_TOKENS` tokens (otherwise the model may stop early at an end-of-sequence token, and your timings won't be comparable).

### 6a. A fixed prompt

In [ ]:
torch.manual_seed(0)
BASE_TEXT = ("The history of computing is a story of moving data closer to where it is processed. "
             "From vacuum tubes to transistors, from mainframes to GPUs, each generation of hardware ")
def make_prompt_ids(prompt_len: int) -> torch.Tensor:
    ids = tok(BASE_TEXT * 50, return_tensors="pt").input_ids[:, :prompt_len]
    assert ids.shape[1] == prompt_len
    return ids.to("cuda")

input_ids = make_prompt_ids(PROMPT_LEN)
print(input_ids.shape)

### 6b. Timing helpers
✏️ Write a timer for one call of a function, and a helper that does warmup + repeated runs.

In [ ]:
def time_ms(fn) -> float:
    # Run fn() once and return its wall time in milliseconds, measured correctly for GPU work.
    raise NotImplementedError

def repeat(fn, warmup: int = 3, runs: int = 10) -> tuple[float, float]:
    # Call fn() `warmup` times (discard), then `runs` times. Return (mean_ms, std_ms).
    raise NotImplementedError

In [ ]:
# CHECK: timer catches async GPU work
big = torch.randn(4096, 4096, device="cuda", dtype=DTYPE)
def work():
    for _ in range(20):
        big @ big
t = time_ms(work)
expected_min = 20 * matmul_flops(4096, 4096, 4096) / (PEAK_BF16_TFLOPS * 1e12) * 1000
print(f"20 matmuls took {t:.2f} ms  (roofline minimum ≈ {expected_min:.2f} ms)")
assert t >= 0.8 * expected_min, "Faster than the hardware allows: you're not synchronizing."
m, s = repeat(work)
assert s < 0.3 * m, f"Very noisy ({m:.2f} ± {s:.2f} ms). Is something else using the GPU?"
print(f"✅ timer OK: {m:.2f} ± {s:.2f} ms  ->  {20*matmul_flops(4096,4096,4096)/(m/1000)/1e12:.1f} TFLOP/s achieved")
del big

✍️ How close did the matmul get to your `PEAK_BF16_TFLOPS`? If it's well below, why might that be (think: laptop power limits, clocks, whether the number you found is boost or base)? If it's *above*, your spec number is wrong.

*Your answer:*

### 6c. Benchmark generation
✏️ Write `benchmark_generate`. It should return a dict with at least: `ttft_ms`, `decode_ms_per_token`, `decode_tok_s`, `peak_mem_gb`, each a mean, plus the matching `_std` values for the timing ones.

In [ ]:
def generate_n(input_ids, n_new: int):
    # Greedy-generate exactly n_new tokens with Hugging Face. Return the output ids.
    raise NotImplementedError

def benchmark_generate(input_ids, n_new: int, warmup: int = 3, runs: int = 10) -> dict:
    raise NotImplementedError

<details><summary>💡 Hint 1</summary>
Look at the <code>generate</code> arguments <code>max_new_tokens</code>, <code>min_new_tokens</code>, and <code>do_sample</code>. Wrap calls in <code>torch.no_grad()</code> (or <code>torch.inference_mode()</code>).
</details>
<details><summary>💡 Hint 2</summary>
<code>generate</code> doesn't tell you when the first token came out. One clean trick: time generating 1 token (≈ TTFT) and generating N tokens. The difference is the time for the remaining N−1 decode steps.
</details>
<details><summary>💡 Hint 3</summary>
<code>decode_ms_per_token = (T(N) − T(1)) / (N − 1)</code>. For the std of a difference, you can either repeat the pair together and take the std of the per-run differences, or report the std of each part separately. Say which you did.
</details>

In [ ]:
# CHECK: generation length and sanity of results
out_ids = generate_n(input_ids, NEW_TOKENS)
assert out_ids.shape[1] == PROMPT_LEN + NEW_TOKENS, f"Expected exactly {NEW_TOKENS} new tokens, got {out_ids.shape[1] - PROMPT_LEN}."
print("sample continuation:", repr(tok.decode(out_ids[0, PROMPT_LEN:PROMPT_LEN+30])))

res = benchmark_generate(input_ids, NEW_TOKENS)
for k, v in res.items():
    print(f"  {k:24s} {v:10.3f}")
for k in ["ttft_ms", "decode_ms_per_token", "decode_tok_s", "peak_mem_gb"]:
    assert k in res, f"missing key {k}"
assert res["peak_mem_gb"] * 1e9 >= weight_bytes(cfg), "Peak memory below the weight size? Check reset_peak_memory_stats / units."
assert res["decode_ms_per_token"] >= 0.95 * pred_decode, (
    "Measured decode is faster than the memory-bandwidth limit. That's impossible: check timing or your bandwidth number.")
assert abs(res["decode_tok_s"] - 1000 / res["decode_ms_per_token"]) / res["decode_tok_s"] < 0.05
print(f"\n✅ baseline measured.  measured / predicted decode time = {res['decode_ms_per_token']/pred_decode:.1f}x")

✍️ **Explain the gap.** Your measured decode time is probably several times slower than the bandwidth prediction. For a model this small at batch size 1, why? Some ideas to investigate (pick at least one and test it):
- How many separate GPU kernels does one decode step launch? (Rough count: layers × ops per layer.) Each launch costs a few microseconds of CPU time. Does that add up to your gap?
- Is the GPU actually busy? Watch `nvidia-smi` utilization while running, or try `torch.profiler`.
- What happens to tokens/sec if you double the prompt length? Halve `NEW_TOKENS`?

This "the model is too small to saturate the GPU; Python and kernel-launch overhead dominate" lesson is exactly what Step 2 (and later CUDA graphs / `torch.compile`) attacks.

*Your answer:*

### 6d. Record results

📖 Every benchmark you run in this project goes into one table, in order, so the story of each speedup is visible. This is a nice first use of pandas.

✏️ Build a dict for this run with the result numbers **and** the context needed to reproduce it: model, dtype, batch size, prompt length, new tokens, GPU name, torch version, transformers version, driver version, date, and a note (e.g. "HF baseline, plugged in, performance mode"). Append it to `results/benchmarks.csv` (create the file if it doesn't exist), then print the table as markdown to paste into your README.

In [ ]:
import pandas as pd, transformers, datetime, subprocess

def record_result(res: dict, note: str, path: str = "../results/benchmarks.csv") -> pd.DataFrame:
    raise NotImplementedError

df = record_result(res, note="HF generate baseline, bs=1")
print(df.tail())

<details><summary>💡 Hint 1</summary>
<code>pd.DataFrame([row])</code> makes a one-row table. <code>df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)</code> appends. <code>df.to_markdown()</code> needs the <code>tabulate</code> package. Driver version: <code>nvidia-smi --query-gpu=driver_version --format=csv,noheader</code>.
</details>

### 6e. Reproducibility check
Restart the kernel, run everything again, and compare with the first run. **Done when** the two runs' means are within each other's spread. If not, find out why (thermals? power mode? something else on the GPU?) before going further.

## 7. Modal: the same benchmark in the cloud

📖 Modal runs your Python function on a cloud GPU. You define a container image and a function; `.remote()` runs it in the cloud. Read first:
- Modal docs: *Getting started*, *GPU acceleration* (`gpu=` argument), *Images* (installing packages), and the page on **cold start** performance.
- Modal's **pricing** page, to get the per-second price of the GPU you pick.

📖 **Cold start** = time from your call until your function actually starts, when no container is running (container boot + image pull + Python imports + model loading). It matters a lot for serverless inference, because users pay it in latency and you pay for it in GPU-seconds.

✏️ Create `bench/modal_bench.py` (a script, not in the notebook). Requirements:
1. An image with torch (cu128 or whatever matches the Modal GPU), transformers, accelerate.
2. A GPU function that loads the model and runs **the same benchmark** as above (move your helpers into a shared file such as `bench/common.py` so both use identical code). It returns the results dict plus timestamps for when the function started and when the model finished loading.
3. A local entrypoint that records the time before calling `.remote()` and computes:
   - **cold start** = function start time − call time (first call, no warm container),
   - **model load time** separately,
   - the same TTFT / decode numbers,
   - **cost per million generated tokens** = (GPU price per second) ÷ (decode tokens per second) × 1e6.
4. Pick one GPU (an L4 or A10 is cheap and good for comparison). Keep runs short; you have $30/month in credits. Make sure the app stops when done.

Then add the Modal row(s) to `results/benchmarks.csv`.

✍️ Predict before you run: using the Modal GPU's spec bandwidth, what decode speed does the roofline give? Will Modal be faster or slower than your laptop, and why?

<details><summary>💡 Hint 1</summary>
Decorate a function with <code>@app.function(gpu="L4", image=image, timeout=600)</code>; call it from a function decorated with <code>@app.local_entrypoint()</code>; run with <code>modal run bench/modal_bench.py</code>. Clock skew between your laptop and Modal can be a few ms; for a cold start of several seconds that's fine, but mention it.
</details>
<details><summary>💡 Hint 2</summary>
To get a fair cold start, make sure no container is alive from a previous run (they stay warm for a while after a call). Measure a second call right after to see the warm-start number for contrast.
</details>

## 8. README

Update your repo's README with:
1. One paragraph: what you're building and why.
2. Your hardware + software versions.
3. The benchmark table (predicted vs measured, laptop vs Modal).
4. Two or three sentences explaining the gap between predicted and measured decode speed.

Commit and push. That's Step 1's deliverable.

## 9. Quiz ❓

Answer these in your own words (out loud or written) without looking at notes. Bring your answers to the project thread and I'll check them and follow up on anything shaky.

1. Why is decode at batch size 1 memory-bound while prefill of a long prompt is compute-bound? Use arithmetic intensity in your answer.
2. Your GPU has 8 GB. Roughly how many total tokens of KV cache can you fit next to the weights? How does GQA change that number, and what would it be without GQA?
3. Your measured decode speed was slower than the roofline prediction. Give two concrete reasons and how you'd test each.
4. Why must you call `torch.cuda.synchronize()` (or use CUDA events) when timing, and what goes wrong if you don't?
5. What is a cold start on Modal made of, and which parts could you shrink?